# Visualizing Isolated Nodes during Cross-Graph Connection
When generating meshes for highly irregular or sparse grids, restrictive distance parameters can fail to connect certain nodes, causing them to be isolated graph components.
This notebook demonstrates how the warning alerts users to this issue.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from loguru import logger
import networkx as nx

import weather_model_graphs as wmg
from weather_model_graphs.visualise import nx_draw_with_pos_and_attr

# Set up mesh nodes (a dense cluster)
xy_mesh = np.array([
    [0.1, 0.1], [0.1, 0.2], [0.2, 0.1], [0.2, 0.2]
])
# Set up grid nodes (one close, one far away to act as 'sparse' or 'orphaned' node)
xy_grid = np.array([
    [0.15, 0.15], # Close, will connect
    [0.70, 0.70], # Far, will be isolated
])


In [ ]:
# Create base graphs
G_source = wmg.create.mesh.create_single_level_2d_mesh_graph(xy=xy_mesh, nx=2, ny=2)
G_target = wmg.create.grid.create_grid_graph_nodes(xy=xy_grid)

# Plot initial setup before connection
fig, ax = plt.subplots(figsize=(6, 6))
pos_source = {node: G_source.nodes[node]["pos"] for node in G_source.nodes()}
pos_target = {node: G_target.nodes[node]["pos"] for node in G_target.nodes()}

nx.draw_networkx_nodes(G_source, pos=pos_source, ax=ax, node_color='blue', node_size=100, label='Mesh Nodes (Source)')
nx.draw_networkx_nodes(G_target, pos=pos_target, ax=ax, node_color='red', node_size=100, label='Grid Nodes (Target)')
ax.legend()
plt.title('Initial Nodes Setup')
plt.show()


In [ ]:
# Connect with a restrictive max_dist
# We expect a warning triggered here because the far grid node is isolated.
G_connect = wmg.create.base.connect_nodes_across_graphs(
    G_source=G_source,
    G_target=G_target,
    method="within_radius",
    max_dist=0.3, # This is too restrictive to reach the node at [0.70, 0.70]
)


In [ ]:
# Visualizing the final connected graph
fig, ax = plt.subplots(figsize=(6, 6))
# For connected bipartite graph visualization
nx.draw_networkx(G_connect, pos={n: G_connect.nodes[n]['pos'] for n in G_connect.nodes}, 
                 ax=ax, with_labels=False, node_size=100, 
                 node_color=['blue' if n in G_source else 'red' for n in G_connect.nodes()])
plt.title('Connected Graph (Showing Isolated Node due to restrictive max_dist)')
plt.show()

# Observe that the top right node has no incoming edges.
